# 📈 12장 · 작년 이맘때처럼: 시계열 분석 (개정판 실습 노트북)

『빅데이터분석 with 파이썬』 12장의 **05. 파이썬 실습**, **04절 예제의 파이썬 확인**, **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 12-1, 12-5, 12-6, 12-7, 12-8, 12-9 |
| ☆ 선택 | 자습이나 심화 | 12-2, 12-3, 12-4 |

**데이터**
- 국제선 항공 승객: 1949~1960년 월별 144개월(Box–Jenkins의 Series G). GitHub 저장소 `jbrownlee/Datasets`에서 바로 읽습니다.
- 독일 전력 수요: Open Power System Data의 하루 단위 파일, 2006~2017년 4,383일. GitHub 저장소 `jenfly/opsd`에서 바로 읽습니다.
- 미니 프로젝트: 한국전력거래소 "시간별 전국 전력수요량" 2021~2025년(출처 한국전력거래소, 공공데이터포털, 공공누리). 다섯 해를 합친 파일 `kpx_시간별_전국_전력수요량_2021_2025.csv`는 강의 자료에 들어 있습니다. 코랩 왼쪽 파일 창에 올려 두거나, 미니 프로젝트 1단계 앞의 셀로 올립니다. 포털에서 해마다의 파일을 직접 받아 합치는 ☆ 셀도 있습니다(받은 파일이 없으면 건너뜁니다).

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다. 프로펫을 쓰는 실습(12-7, 12-8)은 몇십 초씩 걸립니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**, 시계열 예측 라이브러리 **프로펫(Prophet)**, 나라별 공휴일 목록 **holidays**를 설치합니다. 코랩에 이미 있으면 바로 넘어갑니다. 판다스, 스태츠모델, 사이킷런, 맷플롯립은 코랩에 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib prophet holidays

**데이터 받기** (깃허브 저장소판에서 덧붙인 셀) — 아래 셀은 이 장이 쓰는 강의 자료 파일을 저장소의 `data` 폴더에서 코랩으로 내려받습니다.

In [ ]:
# 이 장의 강의 자료 파일을 책 깃허브 저장소에서 받아 옵니다.
# (파일을 코랩 왼쪽 📁 파일 창에 직접 끌어다 올렸다면 이 셀은 건너뛰어도 됩니다.)
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch12/data/"
for f in [
    "kpx_시간별_전국_전력수요량_2021_2025.csv",
]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
    print("준비됨:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

## 05. 파이썬 실습: 비행기 승객에서 전력 수요까지

이 절의 실습은 코랩(Colab)에서 위에서부터 차례로 실행합니다. 데이터는 모두 인터넷 주소에서 바로 읽습니다. 시계열 모델은 스태츠모델과 프로펫으로, 다음 날 예측은 사이킷런으로 만듭니다. 코랩에 프로펫이나 holidays가 없으면 `!pip install prophet holidays`로 먼저 설치합니다. ★는 수업 시간에 꼭 돌리는 실습이고, ☆는 자습용입니다.

#### [파이썬 실습 12-1 ★] 항공 승객 불러오기와 이동평균

항공 승객 데이터를 날짜 인덱스로 읽고, 12개월 이동평균으로 추세를 구하시오.

① 한글 글꼴을 설정하고, 예측을 채점할 함수 `mae( )`를 만들고, 데이터를 읽습니다. ② 연도별 평균과 해마다 가장 붐빈 달을 셉니다. ③ 무게 13개짜리 창을 밀며 추세를 구하고 그립니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")                  # 라이브러리의 경고 문구는 숨긴다
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib                    # 코랩: 한글 글꼴을 자동으로 설정
except ImportError:                                # 나눔고딕 파일이 이미 있는 컴퓨터
    import glob, os
    import matplotlib.font_manager as fm
    for f in glob.glob(os.path.expanduser("~/.fonts/Nanum*.ttf")):
        fm.fontManager.addfont(f)
    plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

def mae(a, b):                                      # 예측 채점: 평균 절대 오차 (식 12-5)
    return round(float(np.mean(np.abs(np.asarray(a) - np.asarray(b)))), 1)

URL_AIR = ("https://raw.githubusercontent.com/jbrownlee/Datasets/"
           "master/airline-passengers.csv")
air = pd.read_csv(URL_AIR, parse_dates=["Month"], index_col="Month")
y = air["Passengers"].asfreq("MS").astype(float)   # 매달 첫날(MS)을 시각으로
print(len(y), "개월:", y.index[0].strftime("%Y-%m"), "~", y.index[-1].strftime("%Y-%m"))

year_mean = y.groupby(y.index.year).mean()
print("1949년 평균", round(year_mean[1949], 1), "→ 1960년 평균", round(year_mean[1960], 1))
print("해마다 가장 붐빈 달:", y.groupby(y.index.year).idxmax().dt.month.value_counts().to_dict())

k = np.r_[0.5, np.ones(11), 0.5] / 12             # 무게 13개, 양 끝은 절반 (합은 1)
trend = y.rolling(13, center=True).apply(lambda w: w @ k)  # 창을 밀며 무게와 내적
print("무게의 합:", k.sum())
print("추세 1949-07:", round(trend["1949-07-01"], 1), " 1960-06:", round(trend["1960-06-01"], 1))

plt.figure(figsize=(7, 3))
plt.plot(y, color="#8EC5E8", label="승객 수")
plt.plot(trend, color="#E04B3C", label="12개월 이동평균(추세)")
plt.ylabel("승객(천 명)")
plt.legend()
plt.show()

`asfreq("MS")`는 "매달 첫날" 간격을 정해 줍니다. 빠진 달이 있으면 빈칸으로 드러납니다. 이 데이터는 144개월이 빠짐없이 있습니다. 추세는 1949년 7월 126.8천 명에서 1960년 6월 475.0천 명으로 올랐습니다. 양 끝 여섯 달은 창이 모자라 추세가 비어 있습니다. 맨 위의 `mae( )`는 뒤의 실습에서 예측을 채점할 함수입니다(식 12-5).

> **[코드 읽기] `rolling(13, center=True).apply(lambda w: w @ k)`**
>
> `rolling(13, center=True)`는 값 13개를 덮는 창을 가운데에 맞추어 한 칸씩 밉니다. `apply( )`는 창마다 괄호 안의 함수를 부릅니다. `lambda w: w @ k`는 "창의 값 w를 받아 무게 k와 내적한다"는 한 줄짜리 함수입니다. `@`는 2장에서 본 내적 기호입니다. 무게가 모두 같다면 `rolling(12).mean()`처럼 간단히 써도 됩니다.

#### [파이썬 실습 12-2 ☆] 계절 지수와 정상성 검정

스태츠모델의 분해 함수로 달별 계절 지수를 구하고, 로그와 차분을 거칠 때마다 ADF 검정(시계열이 고른지 시험하는 방법)의 p값을 구하시오.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

dec = seasonal_decompose(y, model="multiplicative", period=12)  # 추세 × 계절 × 잔차
season = dec.seasonal.iloc[:12]
season.index = season.index.month
print("달별 계절 지수:", season.round(3).to_dict())
print("두 추세의 가장 큰 차이:", round((dec.trend - trend).abs().max(), 6))

ly = np.log(y)
steps = {"원자료": y, "로그": ly, "로그+1차 차분": ly.diff(),
         "로그+1차+계절 차분": ly.diff().diff(12)}
for name, s in steps.items():
    p = adfuller(s.dropna(), autolag="AIC")[1]
    print(f"{name:12s} ADF p값 {p:.4f}")

`seasonal_decompose( )`가 구한 추세는 [파이썬 실습 12-1]의 추세와 똑같습니다(차이 0.0). 이 함수도 같은 무게 13개로 이동평균을 내기 때문입니다. ADF 검정은 "이 시계열은 고르지 않다(정상성이 없다)"는 가정을 시험합니다. p값이 작을수록 그 가정을 버리고 "고르다"고 볼 수 있습니다. p값이 0.05보다 작아진 것은 계절 차분까지 한 뒤였습니다(0.0002).

#### [파이썬 실습 12-3 ☆] 네 가지 기준선과 겨루기

1958년까지로 배우고 1959~1960년 24개월을 네 가지 기준선으로 예측해 MAE를 구하시오.

① 학습 기간과 시험 기간을 시간 순서로 나눕니다. ② 기준선 넷을 만듭니다. ③ MAE를 구해 막대그래프로 그립니다.

In [ ]:
train, test = y.loc[:"1958"], y.loc["1959":]     # 1958년까지 배우고 1959~1960년(24달) 시험
h = len(test)
mean_f = np.repeat(train.mean(), h)                 # 지금까지의 평균
naive = np.repeat(train.iloc[-1], h)                # 마지막 값 그대로
slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
drift = train.iloc[-1] + slope * np.arange(1, h + 1)  # 첫 값과 끝 값을 이은 기울기로 연장
snaive = np.tile(train.iloc[-12:].values, 2)       # 작년 같은 달

base = {"평균": mean_f, "직전 값": naive, "추세(drift)": drift, "작년 같은 달": snaive}
for name, f in base.items():
    print(f"{name:10s} MAE {mae(test, f):6.1f} (천 명)")

pd.Series({k: mae(test, f) for k, f in base.items()}).plot.barh(
    color=["#8EC5E8", "#8EC5E8", "#8EC5E8", "#E04B3C"], figsize=(6, 2.6))
plt.xlabel("평균 오차 MAE(천 명)")
plt.show()

`np.tile( )`은 배열을 정해진 횟수만큼 이어 붙입니다. 1958년 열두 달을 두 번 이어 24개월 예측을 만들었습니다. "작년 같은 달"이 71.2천 명으로 가장 좋았습니다. 계절을 아는 단순한 방법이 추세만 아는 방법(91.6)보다 강합니다. 이 데이터에서 모델이 넘어야 할 선은 71.2입니다.

#### [파이썬 실습 12-4 ☆] 항공사 모델, 홀트-윈터스, 기준점 옮기기

로그를 씌운 승객 수에 항공사 모델을, 원래 승객 수에 홀트-윈터스를 맞추시오. 1955~1960년을 해마다 그 전해까지로 배워 예측하고, 두 모델과 결합 예측의 MAE를 견주시오. [파이썬 실습 12-3]에서 나눈 학습·시험 기간을 이어서 씁니다.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def airline(series):                                # 박스와 젠킨스의 항공사 모델
    return SARIMAX(np.log(series), order=(0, 1, 1),
                   seasonal_order=(0, 1, 1, 12)).fit(disp=False)

def holt(series):                                   # 홀트-윈터스 (추세 + 곱셈 계절)
    return ExponentialSmoothing(series, trend="add", seasonal="mul",
                                seasonal_periods=12).fit()

sar = airline(train)
p_sar = np.exp(sar.forecast(h))                     # 로그로 배웠으니 지수로 되돌린다
p_hw = holt(train).forecast(h)
print("항공사 모델 θ =", round(sar.params["ma.L1"], 3), " Θ =", round(sar.params["ma.S.L12"], 3))
print("항공사 모델 MAE", mae(test, p_sar), "/ 홀트-윈터스 MAE", mae(test, p_hw))
print("1959년 1~3월 항공사 모델 예측:", p_sar.iloc[:3].round(1).tolist())

rows = []
for Y in range(1955, 1961):                         # 기준점을 한 해씩 옮긴다
    tr, te = y.loc[:str(Y - 1)], y.loc[str(Y)]
    f_sn = tr.iloc[-12:].values
    f_sar = np.exp(airline(tr).forecast(12)).values
    f_hw = holt(tr).forecast(12).values
    rows.append({"시험 해": Y, "작년 같은 달": mae(te, f_sn), "SARIMA": mae(te, f_sar),
                 "홀트-윈터스": mae(te, f_hw), "둘의 평균": mae(te, (f_sar + f_hw) / 2)})
roll = pd.DataFrame(rows).set_index("시험 해")
print(roll.to_string())
print("해마다 1등:", roll.iloc[:, :3].idxmin(axis=1).value_counts().to_dict())
print("여섯 해 평균:", roll.mean().round(1).to_dict())

fc = sar.get_forecast(h).conf_int(alpha=0.05)       # 1959~1960년의 95% 예측 구간
lo, hi = np.exp(fc.iloc[:, 0]).values, np.exp(fc.iloc[:, 1]).values
print("95% 구간 안에 든 달:", int(((test.values >= lo) & (test.values <= hi)).sum()), "/", h)
f61 = airline(y).get_forecast(12)                   # 1960년까지 모두 써서 1961년을 내다본다
mid, ci = np.exp(f61.predicted_mean), np.exp(f61.conf_int(alpha=0.05))
print("1961년 1월", round(mid.iloc[0], 1), ci.iloc[0].round(1).tolist(),
      "/ 7월", round(mid.iloc[6], 1), ci.iloc[6].round(1).tolist())

로그로 배운 모델의 예측은 `np.exp( )`로 되돌려야 승객 수가 됩니다. 1959~1960년 한 번의 시험에서는 홀트-윈터스(29.0)가 항공사 모델(39.4)을 이겼습니다. 그런데 여섯 해를 차례로 시험하니 SARIMA가 세 번, 홀트-윈터스가 두 번, 작년 같은 달이 한 번 이겼습니다. 두 모델의 평균은 한 번도 1등을 하지 않았지만 여섯 해 평균(14.5)은 가장 좋았습니다. 3.6의 결론과 같습니다. 1959~1960년의 실제 24개월은 모두 항공사 모델의 95% 예측 구간 안에 들었습니다.

#### [파이썬 실습 12-5 ★] 독일 전력 데이터 불러오고 계단 찾기

독일 하루 전력 데이터를 읽어 빈칸을 확인하고, 연도별 변화율과 2014년의 달별 변화를 구하시오.

① 날짜 인덱스로 읽고 하루 간격을 정합니다. ② 빈칸과 가장 적은 날, 많은 날을 봅니다. ③ 연도별 평균의 변화율과 2014년의 달별 변화를 구합니다.

In [ ]:
URL_EL = ("https://raw.githubusercontent.com/jenfly/opsd/"
          "master/opsd_germany_daily.csv")
el = pd.read_csv(URL_EL, parse_dates=["Date"], index_col="Date").asfreq("D")
c = el["Consumption"]                               # 하루 전력 소비, 단위 GWh
print(el.shape, el.index[0].date(), "~", el.index[-1].date())
print("빈칸:", el.isna().sum().to_dict())
print("가장 적은 날", c.idxmin().date(), round(c.min(), 1), "/ 가장 많은 날", c.idxmax().date(), round(c.max(), 1))

ym = c.groupby(c.index.year).mean()
change = (ym.pct_change() * 100).round(1)
print("전년 대비 변화(%):", change.loc[2009:2015].to_dict())
m = c.resample("MS").mean()                         # 달마다 평균
ratio = (m.loc["2014"].values / m.loc["2013"].values - 1) * 100
print("2014년 달마다 전년 같은 달 대비(%):", np.round(ratio, 1))

ym.plot.bar(color=["#E04B3C" if k == 2014 else "#8EC5E8" for k in ym.index], figsize=(7, 3))
plt.ylim(1200, 1420)
plt.ylabel("하루 평균 수요(GWh)")
plt.xlabel("")
plt.show()

가장 적은 날은 2009년 부활절 일요일이었고, 가장 많은 날은 12월 초의 평일이었습니다. 이 두 날만 봐도 무엇이 수요를 움직이는지 짐작할 수 있습니다. `pct_change( )`는 바로 앞 해와 견준 변화율입니다. `resample("MS").mean( )`은 하루 값을 달마다 평균합니다. 2014년의 8.8%는 열두 달 모두에 고르게 나타나 3.4의 계단을 이룹니다.

> **[코드 읽기] `resample( )`과 `pct_change( )`**
>
> `resample("MS")`는 날짜 인덱스를 "달의 첫날" 단위로 묶습니다. 뒤에 `.mean( )`을 붙이면 달마다 평균을, `.sum( )`을 붙이면 합을 냅니다. `groupby(c.index.year)`처럼 쓸 수도 있지만, `resample( )`은 빈 달도 빠뜨리지 않고 시간 순서대로 줄을 세웁니다. `pct_change( )`는 (이번 값 ÷ 앞 값 − 1)을 계산합니다.

#### [파이썬 실습 12-6 ★] 요일, 달, 자기상관

요일별, 달별 평균 수요와 여러 시차의 자기상관을 구하시오.

① 요일별 평균과 주말의 감소율을 구합니다. ② 달별 평균을 구합니다. ③ 시차 1~7, 14, 364, 365일의 자기상관을 구해 그립니다.

In [ ]:
dow = c.groupby(c.index.dayofweek).mean()          # 0=월요일 … 6=일요일
print("요일 평균(월~일):", dow.round(1).tolist())
print("주말은 평일보다", round((1 - dow[[5, 6]].mean() / dow[:5].mean()) * 100, 1), "% 적다")
mon = c.groupby(c.index.month).mean()
print("가장 많은 달", mon.idxmax(), "월", round(mon.max(), 1), "/ 가장 적은 달", mon.idxmin(), "월", round(mon.min(), 1))

lags = [1, 2, 3, 4, 5, 6, 7, 14, 364, 365]
r = pd.Series({k: c.autocorr(k) for k in lags})  # 시차 k만큼 민 자기 자신과의 상관계수
print("자기상관:", r.round(3).to_dict())

r.plot.bar(color=["#E04B3C" if k in (7, 14, 364) else "#8EC5E8" for k in lags], figsize=(7, 3))
plt.xlabel("시차(일)")
plt.ylabel("자기상관")
plt.xticks(rotation=0)
plt.show()

`dayofweek`는 월요일이 0, 일요일이 6입니다. `c.autocorr(k)`는 c와 k칸 민 c의 상관계수입니다. 시차 6과 7이 높고 2~4가 낮은 것은 한 주의 모양 때문입니다. 365일 시차가 364일보다 크게 낮은 것은 요일이 하루씩 어긋나기 때문입니다.

> **[코드 읽기] `shift(k)`**
>
> `c.shift(7)`은 모든 값을 7칸 뒤로 밉니다. 그러면 오늘 자리에 지난주 같은 요일의 값이 놓이고, 맨 앞 일곱 칸은 빈칸이 됩니다. 시계열의 지연 특징은 거의 모두 `shift( )` 하나로 만듭니다. 다만 [파이썬 실습 12-7]처럼 시험 기간 전체를 한꺼번에 예측할 때는 민 값이 시험 기간 안에서 오지 않는지 확인해야 합니다.

#### [파이썬 실습 12-7 ★] 2017년을 기준선과 프로펫으로 예측하기

2016년까지로 배우고 2017년 365일을 예측하시오. 작년 같은 요일, 휴일을 모르는 프로펫, 휴일을 알려 준 프로펫을 견주고 가장 크게 틀린 날을 찾으시오.

① 시험 기간의 값이 섞이지 않게 "작년 같은 요일"을 만듭니다. ② 프로펫을 휴일 없이, 휴일과 함께 두 번 맞춥니다. ③ 가장 크게 틀린 날과 예측 구간을 확인합니다.

In [ ]:
import logging
from prophet import Prophet
logging.getLogger("cmdstanpy").disabled = True    # 학습 진행 메시지를 숨긴다

tr_e, te_e = c.loc[:"2016"], c.loc["2017"]          # 2016년까지 배우고 2017년 365일 시험
f_mean = np.repeat(tr_e.loc["2016"].mean(), 365)
f_364 = np.tile(tr_e.iloc[-364:].values, 2)[:365]   # 52주 전 같은 요일 (2017년 값은 쓰지 않는다)
leak = c.shift(364).loc["2017"]                     # 흔한 실수: 시험 해의 값이 섞인다
print("shift(364)가 12월 31일 자리에 넣은 값의 날짜:", (te_e.index[-1] - pd.Timedelta(days=364)).date())
print("2016년 평균   MAE", mae(te_e, f_mean))
print("작년 같은 요일 MAE", mae(te_e, f_364), " (shift(364)로 하면", mae(te_e, leak), ")")

def fit_prophet(series, holidays_on, days=365):
    df = series.reset_index()
    df.columns = ["ds", "y"]                        # 프로펫은 열 이름이 ds, y여야 한다
    m = Prophet(daily_seasonality=False)
    if holidays_on:
        m.add_country_holidays(country_name="DE")   # 독일 공휴일 달력을 알려 준다
    m.fit(df)
    future = m.make_future_dataframe(periods=days)
    np.random.seed(0)                               # 예측 구간은 무작위 표본으로 만든다
    return m, m.predict(future).set_index("ds")

m0, fc0 = fit_prophet(tr_e, False)
m1, fc1 = fit_prophet(tr_e, True)
p0, p1 = fc0["yhat"].loc["2017"], fc1["yhat"].loc["2017"]
print("프로펫 기본   MAE", mae(te_e, p0))
print("프로펫 + 휴일 MAE", mae(te_e, p1))
err = (te_e - p1).abs().sort_values(ascending=False)
d0 = err.index[0]
print("가장 크게 틀린 날:", d0.date(), " 실제", round(te_e[d0], 1), " 예측", round(p1[d0], 1))
print("일주일 전 화요일", round(c["2017-10-24"], 1), "/ 10월 3일(통일의 날)", round(c["2017-10-03"], 1))
print("배운 휴일:", list(m1.train_holiday_names))
inside = ((te_e >= fc1["yhat_lower"].loc["2017"]) & (te_e <= fc1["yhat_upper"].loc["2017"])).mean()
print("80% 예측 구간 안에 든 날:", round(inside * 100, 1), "%")

plt.figure(figsize=(7, 3))
plt.plot(te_e.loc["2017-10"], color="black", label="실제")
plt.plot(p1.loc["2017-10"], color="#E04B3C", label="프로펫 + 휴일")
plt.plot(pd.Series(f_364, te_e.index).loc["2017-10"], color="#8EC5E8", label="작년 같은 요일")
plt.ylabel("GWh")
plt.legend()
plt.show()

`np.tile(tr_e.iloc[-364:].values, 2)[:365]`는 2016년의 마지막 52주를 두 번 이어 붙인 뒤 365일만 씁니다. `c.shift(364)`로 만든 예측과는 12월 31일 하루만 다르지만, 시험 기간의 값을 쓰지 않는다는 점이 다릅니다. 프로펫은 열 이름이 `ds`(날짜)와 `y`(값)인 데이터프레임을 받습니다. 배운 휴일 목록에 종교개혁 기념일(Reformation Day)이 없다는 점을 확인하십시오.

80% 예측 구간에 2017년의 90.4%가 들어왔습니다. 좋은 일처럼 보이지만, 80% 구간이라면 80% 안팎이 들어와야 맞습니다. 구간이 실제보다 넓게, 곧 지나치게 조심스럽게 잡혔다는 뜻입니다. 예측 구간은 무작위 표본으로 만들기 때문에 실행할 때마다 끝자리가 조금 달라질 수 있습니다.

#### [파이썬 실습 12-8 ★] 해마다 기준점을 옮겨 전력 수요 예측하기

2012년부터 2017년까지 해마다 그 전해까지로 배워 그해를 예측하고, 작년 같은 요일, 프로펫, 둘의 평균을 견주시오. 프로펫을 여섯 번 맞추므로 몇 분 걸립니다.

In [ ]:
rows = []
for Y in range(2012, 2018):                         # 2012~2017년을 차례로 시험
    tr, te = c.loc[:str(Y - 1)], c.loc[str(Y)]
    sn = np.tile(tr.iloc[-364:].values, 2)[:len(te)]  # 시험 해의 값은 쓰지 않는다
    p = fit_prophet(tr, True, len(te))[1]["yhat"].loc[str(Y)].values
    rows.append({"시험 해": Y, "작년 같은 요일": mae(te, sn), "프로펫+휴일": mae(te, p),
                 "둘의 평균": mae(te, (sn + p) / 2)})
eroll = pd.DataFrame(rows).set_index("시험 해")
print(eroll.to_string())
print("두 방법 가운데 1등:", eroll.iloc[:, :2].idxmin(axis=1).value_counts().to_dict())
print("여섯 해 평균:", eroll.mean().round(1).to_dict())

[표 12-3]의 숫자입니다. 2014년에는 세 방법 모두 크게 틀렸습니다. 2015년에는 프로펫이 2014년의 계단을 추세로 읽어 크게 틀렸습니다. 결합 예측은 어느 해에도 크게 지지 않았습니다.

#### [파이썬 실습 12-9 ★] 지연 특징으로 다음 날 예측하기

어제, 지난주 같은 요일, 작년 같은 요일, 지난 7일 평균, 달, 요일, 휴일을 특징으로 만들어 랜덤 포레스트로 2017년의 매일 다음 날을 예측하시오.

① 독일 공휴일 열을 만듭니다. ② `shift( )`로 지연 특징을 만들고 2016년까지와 2017년으로 나눕니다. ③ 랜덤 포레스트를 배우게 하고 기준선과 견줍니다.

In [ ]:
import holidays
from sklearn.ensemble import RandomForestRegressor

de = holidays.Germany(years=range(2006, 2018))      # 독일 전국 공휴일
hol = pd.Series([d in de for d in c.index.date], index=c.index).astype(int)
wk = c.index.dayofweek < 5                          # 월~금
print("평일 공휴일 평균", round(c[wk & (hol == 1)].mean(), 1), "/ 보통 평일 평균", round(c[wk & (hol == 0)].mean(), 1))
X = pd.DataFrame({"어제": c.shift(1), "지난주 같은 요일": c.shift(7),
                  "작년 같은 요일": c.shift(364), "지난 7일 평균": c.shift(1).rolling(7).mean(),
                  "달": c.index.month, "요일": c.index.dayofweek,
                  "휴일": hol, "어제가 휴일": hol.shift(1, fill_value=0)})
data = X.join(c.rename("y")).dropna()               # 앞의 364일은 작년 값이 없어 빠진다
trB, teB = data.loc[:"2016"], data.loc["2017"]
feats = [k for k in data.columns if k != "y"]
print("학습", len(trB), "일 / 시험", len(teB), "일")
print("어제 값 MAE", mae(teB.y, teB["어제"]), "/ 지난주 같은 요일 MAE", mae(teB.y, teB["지난주 같은 요일"]))

rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=0, n_jobs=-1)
rf.fit(trB[feats], trB.y)
pred = rf.predict(teB[feats])
print("랜덤 포레스트 MAE", mae(teB.y, pred), f"(평균 수요의 {mae(teB.y, pred) / teB.y.mean() * 100:.1f}%)")
imp = pd.Series(rf.feature_importances_, feats).sort_values()
print(imp.sort_values(ascending=False).round(3).head(4).to_string())

imp.plot.barh(color="#E04B3C", figsize=(6, 3))
plt.xlabel("중요도")
plt.show()

지연 특징은 모두 `shift(1)` 이상으로 만들어, 예측하는 날의 값이 특징에 섞이지 않게 했습니다. `rolling(7).mean( )` 앞에도 `shift(1)`을 붙였습니다. 이것을 빼면 오늘 값이 평균에 들어가는 누수가 생깁니다. 랜덤 포레스트는 18.7 GWh로, 지난주 같은 요일(54.1)의 3분의 1 남짓만 틀렸습니다.

> **핵심 3줄**
>
> 시계열 예측은 "작년 같은 요일" 같은 기준선부터 만들고, 프로펫이나 지연 특징 모델이 그것을 이기는지 시간 순서대로 시험합니다.
>
> 한 해만 시험하면 운이 섞이므로 기준점을 해마다 옮겨 되풀이하고, 특징은 예측하는 날보다 앞선 값으로만 만듭니다.
>
> 2017년 한 해는 프로펫 + 휴일(31.6 GWh)이 작년 같은 요일(34.8)을 이겼지만, 여섯 해 평균은 60.2 대 60.0으로 비겼고, 둘의 평균이 53.3으로 가장 나았습니다.

## 04절 예제를 파이썬으로 확인

책 04절의 [예제 12-1]~[예제 12-4]에서 손으로 계산한 값을 파이썬으로 다시 구합니다.

### [예제 12-1] 3개월 이동평균 = 무게와 값의 내적

In [ ]:
w = np.array([1, 1, 1]) / 3                         # 무게(커널) 세 개, 합은 1
print(round(np.array([112, 118, 132]) @ w, 1))      # 내적 = 3개월 이동평균

### [예제 12-2] 엇갈리는 시계열의 자기상관

In [ ]:
s = pd.Series([10, 20, 10, 20, 10])
print("시차 1:", round(s.autocorr(1), 3), " 시차 2:", round(s.autocorr(2), 3))

### [예제 12-3] 지수평활을 손으로 계산하기

In [ ]:
s_t = 112                                           # 첫 평활값
for v in [118, 132]:
    s_t = 0.3 * v + 0.7 * s_t                       # (식 12-4)
    print(round(s_t, 2))

### [예제 12-4] 작년 같은 달과 항공사 모델의 오차 견주기

In [ ]:
real = np.array([360, 342, 406])
for name, f in [("작년 같은 달", [340, 318, 362]), ("항공사 모델", [348.6, 331.6, 383.2])]:
    e = np.abs(real - np.array(f))
    print(f"{name}  MAE {e.mean():.1f}  MAPE {(e / real).mean() * 100:.1f}%")

## [미니 프로젝트 12] 한국에서도 "지난주 이날"이 이길까

**질문.** 독일에서는 "지난주 같은 요일"이 "어제"보다 내일의 전력 수요를 훨씬 잘 맞혔습니다. 여름에는 냉방, 겨울에는 난방으로 전기를 많이 쓰는 한국에서도 그럴까요? 우리나라의 하루 전력 수요로 확인해 봅니다.

**데이터.** 한국전력거래소가 공공데이터포털에 올린 "한국전력거래소_시간별 전국 전력수요량"입니다. 2021년 1월 1일부터 2025년 12월 31일까지 하루가 한 행이고, 열은 날짜와 1시~24시의 수요(MW)로 25개입니다. 24칸을 평균해 24를 곱하면 하루 전력량(MWh)이 되고, 1,000으로 나누면 GWh입니다. 다섯 해를 한 파일로 합친 kpx_시간별_전국_전력수요량_2021_2025.csv가 강의 자료에 들어 있습니다. 이 파일을 Colab 왼쪽 파일 창에 끌어다 놓고 시작합니다. 다른 해를 쓰려면 1단계 맨 위의 세 줄(파일 이름, 날짜 열, 시험할 해)만 바꿉니다.

> **[더 알아보기] ☆ 포털에서 직접 받아 합치기**
>
> 포털에는 해마다 파일이 하나씩 있습니다. 강의 자료의 파일은 2021~2025년 다섯 파일을 차례로 이은 것입니다. 직접 합칠 때는 세 가지를 챙깁니다. 한글이 cp949로 저장되어 있으니 `encoding="cp949"`로 읽습니다. 해마다 열 이름을 같게 맞춘 뒤 잇습니다. 이은 뒤에는 날짜순으로 정렬하고 같은 날짜는 하나만 남깁니다.

In [ ]:
# ☆ 공공데이터포털에서 받은 해마다의 파일을 하나로 합치기 (강의 자료의 파일을 쓰면 건너뜁니다)
import glob
RAW = sorted(glob.glob("한국전력거래소_시간별*전력수요량*.csv"))  # 받은 파일 이름에 맞게 고칩니다
if not RAW:
    print("포털에서 받은 파일이 없습니다. 강의 자료의 합친 파일로 1단계를 하십시오.")
else:
    parts = []
    for f in RAW:
        t = pd.read_csv(f, encoding="cp949")       # 포털 파일의 한글은 cp949
        t.columns = ["날짜"] + [f"{h}시" for h in range(1, 25)]  # 해마다 열 이름을 똑같이
        parts.append(t)
    kp_all = pd.concat(parts, ignore_index=True)    # 위아래로 잇는다
    kp_all = kp_all.dropna(subset=["날짜"])         # 내용 없는 줄을 버린다
    kp_all["날짜"] = pd.to_datetime(kp_all["날짜"])
    kp_all = kp_all.sort_values("날짜").drop_duplicates("날짜", keep="last")
    kp_all.to_csv("kpx_직접_합친_파일.csv", index=False)  # 1단계의 PATH를 이 이름으로
    print(len(RAW), "개 파일 →", kp_all.shape, kp_all["날짜"].min().date(), "~", kp_all["날짜"].max().date())

**1단계: 데이터 신원 확인**([파이썬 실습 12-5] 응용). 파일을 읽어 빈칸, 같은 날짜, 날짜 순서를 확인하고 하루 단위 GWh로 바꾸시오.

In [ ]:
# 파일 창에 kpx_시간별_전국_전력수요량_2021_2025.csv를 올리지 않았다면, 이 셀이 올릴 창을 엽니다.
import os
if not os.path.exists("kpx_시간별_전국_전력수요량_2021_2025.csv"):
    from google.colab import files
    files.upload()

In [ ]:
# 데이터 바꾸기: 다른 해나 다른 파일을 쓸 때는 이 세 줄만 고칩니다.
PATH = "kpx_시간별_전국_전력수요량_2021_2025.csv"   # Colab 왼쪽 파일 창에 올린 파일
DATE = "날짜"                                       # 날짜 열 (나머지 24열은 1시~24시의 수요, MW)
TEST = "2025"                                       # 시험할 해 (그 앞은 모두 학습에 쓴다)

kp = pd.read_csv(PATH)
print(kp.shape, "/ 빈칸", kp.isna().sum().sum(), "/ 날짜가 빈 줄", kp[DATE].isna().sum())
kp = kp.dropna(subset=[DATE])                       # 내용 없는 줄을 버린다
kp[DATE] = pd.to_datetime(kp[DATE])
print("같은 날이 두 번:", kp[DATE].duplicated().sum(), "/ 날짜 순서대로인가:", kp[DATE].is_monotonic_increasing)
kp = kp.set_index(DATE).sort_index()
day = (kp.mean(axis=1) * 24 / 1000).asfreq("D")     # 하루 GWh = 24시간 평균(MW) × 24 ÷ 1000
print(len(day), "일, 빈칸", day.isna().sum())
print("가장 적은 날", day.idxmin().date(), round(day.min(), 1), "/ 가장 많은 날", day.idxmax().date(), round(day.max(), 1))

빈칸 25개는 한 줄에 몰려 있었습니다. 2025년 파일 끝의 빈 줄이 합칠 때 따라 들어온 것입니다. 이 줄을 버리면 1,826일이 빈틈없이 이어집니다(2024년이 윤년이라 365 × 5 + 1). 가장 적은 날은 2023년 10월 1일로, 9월 28일부터 10월 3일까지 이어진 추석 연휴의 한가운데입니다. 가장 많은 날은 2024년 8월 20일, 한여름입니다.

**2단계: 기준선 만들기**([파이썬 실습 12-6] 응용). 시차 1, 7, 364일의 자기상관을 구하고, 2025년의 매일을 "어제 값"과 "지난주 같은 요일"로 예측해 MAE를 견주시오.

In [ ]:
for k in [1, 7, 364]:
    print(f"시차 {k:3d}일 자기상관 {day.autocorr(k):.3f}")
tY = day.loc[TEST]
print(TEST + "년 다음 날 예측 — 어제 값 MAE", mae(tY, day.shift(1).loc[TEST]),
      "/ 지난주 같은 요일 MAE", mae(tY, day.shift(7).loc[TEST]))
print("달별 평균(GWh):", day.groupby(day.index.month).mean().round(0).astype(int).tolist())

독일만큼 뚜렷하지 않습니다. 지난주와의 자기상관(0.802)이 어제(0.765)보다 조금 높을 뿐이고, MAE도 "지난주 같은 요일" 92.3 GWh, "어제 값" 94.5 GWh로 거의 비깁니다. 달별 평균에 까닭이 보입니다. 독일은 겨울에 많고 여름에 적은 산 하나였지만, 한국은 1월과 8월에 산이 둘이고 봄·가을이 골짜기입니다. 더위와 추위는 며칠씩 이어지며 수요를 끌고 다니므로, 요일만큼 어제의 날씨도 큰 몫을 합니다. 둘 가운데 나은 92.3 GWh가 넘어야 할 선입니다.

**3단계: 지연 특징과 랜덤 포레스트**([파이썬 실습 12-9] 응용). holidays.KR로 한국 공휴일(설·추석 연휴와 대체 휴일 포함)을 더해 지연 특징을 만들고, 2024년까지로 배워 2025년의 다음 날을 예측하시오.

In [ ]:
import holidays
from sklearn.ensemble import RandomForestRegressor
kr = holidays.KR(years=range(day.index.year[0], day.index.year[-1] + 1))  # 설·추석 연휴, 대체 휴일까지
hol_k = pd.Series([d in kr for d in day.index.date], index=day.index).astype(int)
Xk = pd.DataFrame({"어제": day.shift(1), "지난주 같은 요일": day.shift(7),
                   "작년 같은 요일": day.shift(364), "지난 7일 평균": day.shift(1).rolling(7).mean(),
                   "달": day.index.month, "요일": day.index.dayofweek,
                   "휴일": hol_k, "어제가 휴일": hol_k.shift(1, fill_value=0)})
dk = Xk.join(day.rename("y")).dropna()
trK, teK = dk.loc[:str(int(TEST) - 1)], dk.loc[TEST]
fk = list(Xk.columns)
rfk = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=0, n_jobs=-1)
rfk.fit(trK[fk], trK.y)
pk = rfk.predict(teK[fk])
print("학습", len(trK), "일 / 시험", len(teK), "일")
print("랜덤 포레스트 MAE", mae(teK.y, pk), f"(평균 수요의 {mae(teK.y, pk) / teK.y.mean() * 100:.1f}%)")
imp_k = pd.Series(rfk.feature_importances_, fk).sort_values(ascending=False)
print(imp_k.round(3).head(3).to_string())

오차가 92.3에서 43.1 GWh(평균 수요의 2.8%)로 절반 아래가 되었습니다. 모델이 가장 기댄 특징은 여기서도 "지난주 같은 요일"입니다. 그래도 독일(1.4%)의 두 배입니다. 배운 날이 독일보다 훨씬 적고(1,097일 대 3,654일), 무엇보다 모델이 기온을 모릅니다.

**4단계: 어디서 틀렸나 보기.** 달별 MAE와 가장 크게 틀린 날 다섯을 구하고, 그날 무슨 일이 있었는지 찾아보시오. 5월 9일이라면 모델이 기댄 "지난주 같은 요일", 곧 5월 2일이 어떤 날이었는지부터 확인합니다.

In [ ]:
e = pd.Series(np.abs(teK.y.values - pk), index=teK.index)
print("달별 MAE:", e.groupby(e.index.month).mean().round(1).tolist())
big = pd.DataFrame({"실제": teK.y, "예측": pk}).loc[e.nlargest(5).index].round(1)  # 가장 크게 틀린 닷새
big["공휴일"] = [kr.get(d, "") for d in big.index.date]
print(big.to_string())

**5단계: 한계 쓰기.** 이 모델이 아직 모르는 것을 [표 12-4]처럼 표로 쓰시오. 기온이 없다는 점은 반드시 다루어야 합니다.

**제출물.** 그림 2장(독일과 한국의 자기상관과 기준선 비교, 2025년 실제와 예측), 성적표 1개(어제 값, 지난주 같은 요일, 랜덤 포레스트의 MAE), 결론 5문장 이내.

**[표 12-5] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 빈 줄, 같은 날짜, 날짜 순서를 확인하고 하루 GWh로 바꾼 방법을 밝힘 | 빈 줄만 지움 | 점검 없이 바로 합계를 냄 |
| 기준선 비교(20%) | 자기상관으로 두 기준선을 고르고 시간 순서로 시험함 | 기준선은 있으나 하나만 씀 | 기준선 없이 모델 점수만 씀 |
| 기법 적용(30%) | 지연 특징을 shift(1) 이상으로 만들고 누수가 없음 | 모델은 돌지만 특징에 그날 값이 섞임 | 코드가 돌지 않거나 결과가 틀림 |
| 해석과 한계(30%) | 독일과 다른 까닭을 달별 모양으로 설명하고, 크게 틀린 날의 원인을 찾아 한계를 표로 씀 | 한계를 한두 가지 씀 | 숫자만 나열함 |

## ✏️ [연습문제 12-6] 휴일 특징을 빼면

[파이썬 실습 12-9]에서 휴일 두 열("휴일", "어제가 휴일")을 빼고 랜덤 포레스트를 다시 배우게 하시오. 2017년 전체와 휴일만의 MAE를 휴일 열이 있을 때와 견주시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
no_hol = [k for k in feats if "휴일" not in k]       # 휴일 두 열을 뺀다
rf2 = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=0, n_jobs=-1)
rf2.fit(trB[no_hol], trB.y)
pred2 = rf2.predict(teB[no_hol])
is_hol = (teB["휴일"] == 1).values
print("한 해 전체 MAE — 휴일 있음", mae(teB.y, pred), "/ 없음", mae(teB.y, pred2))
print("휴일", is_hol.sum(), "일의 MAE — 휴일 있음", mae(teB.y[is_hol], pred[is_hol]),
      "/ 없음", mae(teB.y[is_hol], pred2[is_hol]))